# RSNA Continual Learning - Kaggle Ready

Continual learning for medical images using preprocessed RSNA dataset.

**Methods Implemented:**
- Baseline (Naive Fine-tuning)
- EWC (Elastic Weight Consolidation)
- Experience Replay
- EWC + Replay (Combined)

**Dataset:** RSNA 2023 Abdominal Trauma (preprocessed PNGs)

## 1. Setup and Data Loading

In [ ]:
# Install requirements if needed
# !pip install kagglehub torch torchvision tqdm matplotlib seaborn scikit-learn opencv-python

In [ ]:
# Standard libraries
import os
import numpy as np
import pandas as pd
import copy
import tqdm
from pathlib import Path
import random
from collections import Counter

# Image processing
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Kaggle dataset download
import kagglehub

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Download preprocessed RSNA dataset from Kaggle
path = kagglehub.dataset_download("ashery/rsna-2023-abdominal-trauma-processed-dataset")
print(f"Dataset downloaded to: {path}")

# Set data root
DATA_ROOT = path

## 2. Dataset Classes

Simple dataset classes for loading preprocessed PNG images.

In [ ]:
class RSNADataset(Dataset):
    """
    Dataset for preprocessed RSNA images (PNG format).
    Images are already processed, no CT windowing needed.
    """
    
    def __init__(self, image_paths, labels, target_size=(256, 256), transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.target_size = target_size
        self.transform = transform
        
        assert len(image_paths) == len(labels), "Mismatch between images and labels"
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Failed to load: {img_path}")
        
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize if needed
        if img.shape[:2] != self.target_size:
            img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_LINEAR)
        
        # Convert to tensor (H,W,C) -> (C,H,W) and normalize to [0,1]
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        
        if self.transform:
            img_tensor = self.transform(img_tensor)
        
        return img_tensor, label


class SubDataset(Dataset):
    """Subset containing only specified labels."""
    
    def __init__(self, original_dataset, sub_labels):
        self.original_dataset = original_dataset
        self.sub_labels = sub_labels
        
        # Find indices with desired labels
        self.indices = []
        for idx in range(len(original_dataset)):
            _, label = original_dataset[idx]
            if label in sub_labels:
                self.indices.append(idx)
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        return self.original_dataset[self.indices[idx]]


class MemorySetDataset(Dataset):
    """Dataset from stored samples (for replay)."""
    
    def __init__(self, memory_sets):
        self.memory_sets = memory_sets
    
    def __len__(self):
        return len(self.memory_sets)
    
    def __getitem__(self, idx):
        return self.memory_sets[idx]


def load_rsna_data(data_root, max_samples_per_class=None):
    """
    Load RSNA dataset from directory structure.
    Assumes structure: data_root/train|test/class_X/*.png
    """
    train_paths, train_labels = [], []
    test_paths, test_labels = [], []
    
    # Load training data
    train_dir = os.path.join(data_root, 'train')
    if os.path.exists(train_dir):
        for class_name in sorted(os.listdir(train_dir)):
            class_dir = os.path.join(train_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            
            label = int(class_name.split('_')[-1]) if 'class_' in class_name else int(class_name)
            
            images = [os.path.join(class_dir, f) for f in os.listdir(class_dir) 
                     if f.endswith(('.png', '.jpg'))]
            
            if max_samples_per_class:
                images = images[:max_samples_per_class]
            
            train_paths.extend(images)
            train_labels.extend([label] * len(images))
    
    # Load test data
    test_dir = os.path.join(data_root, 'test')
    if os.path.exists(test_dir):
        for class_name in sorted(os.listdir(test_dir)):
            class_dir = os.path.join(test_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            
            label = int(class_name.split('_')[-1]) if 'class_' in class_name else int(class_name)
            
            images = [os.path.join(class_dir, f) for f in os.listdir(class_dir) 
                     if f.endswith(('.png', '.jpg'))]
            
            if max_samples_per_class:
                images = images[:max_samples_per_class]
            
            test_paths.extend(images)
            test_labels.extend([label] * len(images))
    
    print(f"\nLoaded {len(train_paths)} training images")
    print(f"Loaded {len(test_paths)} test images")
    print(f"Number of classes: {len(set(train_labels))}")
    print(f"Class distribution (train): {Counter(train_labels)}")
    
    return train_paths, train_labels, test_paths, test_labels


def create_task_datasets(base_dataset, num_tasks):
    """Split dataset into class-incremental tasks."""
    # Get all unique labels
    all_labels = set()
    for _, label in base_dataset:
        all_labels.add(label)
    all_labels = sorted(list(all_labels))
    
    classes_per_task = len(all_labels) // num_tasks
    
    task_datasets = []
    for task_id in range(num_tasks):
        start_class = task_id * classes_per_task
        end_class = start_class + classes_per_task
        task_labels = all_labels[start_class:end_class]
        
        task_dataset = SubDataset(base_dataset, sub_labels=task_labels)
        task_datasets.append(task_dataset)
        print(f"Task {task_id+1}: Classes {task_labels}, {len(task_dataset)} samples")
    
    return task_datasets


def fill_memory_buffer(dataset, buffer_size_per_class, class_indices):
    """Fill memory buffer with class-balanced samples."""
    memory_sets = []
    
    for class_id in class_indices:
        class_samples = []
        for idx in range(len(dataset)):
            image, label = dataset[idx]
            if label == class_id:
                class_samples.append((image, label))
        
        if len(class_samples) > buffer_size_per_class:
            sampled = random.sample(class_samples, buffer_size_per_class)
        else:
            sampled = class_samples
        
        memory_sets.extend(sampled)
    
    return memory_sets

In [ ]:
# Load data
train_paths, train_labels, test_paths, test_labels = load_rsna_data(DATA_ROOT)

# Create datasets
train_dataset = RSNADataset(train_paths, train_labels, target_size=(256, 256))
test_dataset = RSNADataset(test_paths, test_labels, target_size=(256, 256))

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")

## 3. Model Architecture

In [ ]:
class MedicalImageClassifier(nn.Module):
    """CNN classifier for medical images."""
    
    def __init__(self, num_classes=2, input_channels=3, dropout_rate=0.5):
        super(MedicalImageClassifier, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        # Feature size after 4 poolings: 256x256 -> 16x16
        self.feature_size = 256 * 16 * 16
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.feature_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        
        x = x.view(-1, self.feature_size)
        
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        
        return x

# Print model info
model = MedicalImageClassifier(num_classes=2, input_channels=3)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel created with {total_params:,} trainable parameters")

## 4. Training and Evaluation Functions

In [ ]:
def train_baseline(model, dataset, iters, lr, batch_size, device):
    """Standard training without continual learning strategies."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    model.to(device)
    
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                            num_workers=0, drop_last=True)
    
    iters_left = 0
    progress_bar = tqdm.tqdm(range(1, iters + 1), desc="Training")
    
    for batch_idx in progress_bar:
        if iters_left == 0:
            data_iter = iter(data_loader)
            iters_left = len(data_loader)
        
        images, labels = next(data_iter)
        images, labels = images.to(device), labels.to(device)
        iters_left -= 1
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})


def evaluate_accuracy(model, dataset, device, batch_size=128):
    """Evaluate model accuracy."""
    model.eval()
    model.to(device)
    
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100.0 * correct / total
    return accuracy

## 5. EWC Implementation

In [ ]:
def estimate_fisher_information(model, dataset, n_samples, device):
    """Estimate Fisher Information Matrix."""
    fisher_info = {}
    for name, param in model.named_parameters():
        if param.requires_grad:
            fisher_info[name.replace('.', '__')] = torch.zeros_like(param.data)
    
    model.eval()
    model.to(device)
    
    data_loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)
    
    n_samples_processed = 0
    for images, labels in data_loader:
        if n_samples_processed >= n_samples:
            break
        
        images = images.to(device)
        
        model.zero_grad()
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        
        for c in range(outputs.size(1)):
            model.zero_grad()
            loss = -torch.log(probs[0, c] + 1e-10)
            loss.backward(retain_graph=True)
            
            for name, param in model.named_parameters():
                if param.requires_grad and param.grad is not None:
                    fisher_info[name.replace('.', '__')] += (
                        probs[0, c].item() * param.grad.data.pow(2)
                    )
        
        n_samples_processed += 1
    
    for name in fisher_info:
        fisher_info[name] /= n_samples_processed
    
    return fisher_info


def train_with_ewc(model, dataset, iters, lr, batch_size, device, current_task, ewc_lambda=5000.0):
    """Train with EWC regularization."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    model.to(device)
    
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                            num_workers=0, drop_last=True)
    
    iters_left = 0
    progress_bar = tqdm.tqdm(range(1, iters + 1), desc=f"Training EWC (Task {current_task})")
    
    for batch_idx in progress_bar:
        if iters_left == 0:
            data_iter = iter(data_loader)
            iters_left = len(data_loader)
        
        images, labels = next(data_iter)
        images, labels = images.to(device), labels.to(device)
        iters_left -= 1
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Add EWC regularization
        if current_task > 1 and hasattr(model, 'fisher_info'):
            ewc_loss = 0
            for name, param in model.named_parameters():
                if param.requires_grad:
                    name_normalized = name.replace('.', '__')
                    if name_normalized in model.fisher_info:
                        fisher = model.fisher_info[name_normalized].to(device)
                        optimal = model.optimal_params[name_normalized].to(device)
                        ewc_loss += (fisher * (param - optimal).pow(2)).sum()
            
            loss += (ewc_lambda / 2) * ewc_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

## 6. Experience Replay

In [ ]:
def train_with_replay(model, dataset, iters, lr, batch_size, device, current_task, buffer_dataset=None):
    """Train with experience replay."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    model.to(device)
    
    current_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=0, drop_last=True)
    
    if buffer_dataset is not None and len(buffer_dataset) > 0:
        replay_loader = DataLoader(buffer_dataset, batch_size=batch_size, shuffle=True, 
                                  num_workers=0, drop_last=True)
        use_replay = True
    else:
        use_replay = False
    
    iters_left_current = 0
    iters_left_replay = 0
    progress_bar = tqdm.tqdm(range(1, iters + 1), desc=f"Training Replay (Task {current_task})")
    
    for batch_idx in progress_bar:
        if iters_left_current == 0:
            current_iter = iter(current_loader)
            iters_left_current = len(current_loader)
        
        images_current, labels_current = next(current_iter)
        images_current = images_current.to(device)
        labels_current = labels_current.to(device)
        iters_left_current -= 1
        
        if use_replay:
            if iters_left_replay == 0:
                replay_iter = iter(replay_loader)
                iters_left_replay = len(replay_loader)
            
            images_replay, labels_replay = next(replay_iter)
            images_replay = images_replay.to(device)
            labels_replay = labels_replay.to(device)
            iters_left_replay -= 1
            
            images = torch.cat([images_current, images_replay], dim=0)
            labels = torch.cat([labels_current, labels_replay], dim=0)
        else:
            images = images_current
            labels = labels_current
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

## 7. Combined EWC + Replay

In [ ]:
def train_with_ewc_replay(model, dataset, iters, lr, batch_size, device, current_task,
                         buffer_dataset=None, ewc_lambda=5000.0):
    """Train with both EWC and replay."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    model.to(device)
    
    current_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=0, drop_last=True)
    
    if buffer_dataset is not None and len(buffer_dataset) > 0:
        replay_loader = DataLoader(buffer_dataset, batch_size=batch_size, shuffle=True, 
                                  num_workers=0, drop_last=True)
        use_replay = True
    else:
        use_replay = False
    
    iters_left_current = 0
    iters_left_replay = 0
    progress_bar = tqdm.tqdm(range(1, iters + 1), desc=f"Training EWC+Replay (Task {current_task})")
    
    for batch_idx in progress_bar:
        if iters_left_current == 0:
            current_iter = iter(current_loader)
            iters_left_current = len(current_loader)
        
        images_current, labels_current = next(current_iter)
        images_current = images_current.to(device)
        labels_current = labels_current.to(device)
        iters_left_current -= 1
        
        if use_replay:
            if iters_left_replay == 0:
                replay_iter = iter(replay_loader)
                iters_left_replay = len(replay_loader)
            
            images_replay, labels_replay = next(replay_iter)
            images_replay = images_replay.to(device)
            labels_replay = labels_replay.to(device)
            iters_left_replay -= 1
            
            images = torch.cat([images_current, images_replay], dim=0)
            labels = torch.cat([labels_current, labels_replay], dim=0)
        else:
            images = images_current
            labels = labels_current
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Add EWC regularization
        if current_task > 1 and hasattr(model, 'fisher_info'):
            ewc_loss = 0
            for name, param in model.named_parameters():
                if param.requires_grad:
                    name_normalized = name.replace('.', '__')
                    if name_normalized in model.fisher_info:
                        fisher = model.fisher_info[name_normalized].to(device)
                        optimal = model.optimal_params[name_normalized].to(device)
                        ewc_loss += (fisher * (param - optimal).pow(2)).sum()
            
            loss += (ewc_lambda / 2) * ewc_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

## 8. Visualization Functions

In [ ]:
def plot_task_accuracies(accuracies, title, task_names=None):
    """Plot accuracy bar chart."""
    num_tasks = len(accuracies)
    if task_names is None:
        task_names = [f'Task {i+1}' for i in range(num_tasks)]
    
    plt.figure(figsize=(8, 5))
    bars = plt.bar(range(num_tasks), accuracies, color='steelblue', alpha=0.7)
    
    for i, (bar, acc) in enumerate(zip(bars, accuracies)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.xlabel('Task', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(range(num_tasks), task_names)
    plt.ylim([0, 105])
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()


def compare_methods(results_dict, task_names=None):
    """Compare accuracy across methods."""
    num_methods = len(results_dict)
    num_tasks = len(list(results_dict.values())[0])
    
    if task_names is None:
        task_names = [f'Task {i+1}' for i in range(num_tasks)]
    
    fig, axes = plt.subplots(1, num_methods, figsize=(5*num_methods, 5))
    if num_methods == 1:
        axes = [axes]
    
    for idx, (method_name, accuracies) in enumerate(results_dict.items()):
        ax = axes[idx]
        bars = ax.bar(range(num_tasks), accuracies, color='steelblue', alpha=0.7)
        
        for bar, acc in zip(bars, accuracies):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Task', fontsize=12)
        ax.set_ylabel('Accuracy (%)', fontsize=12)
        ax.set_title(method_name, fontsize=13, fontweight='bold')
        ax.set_xticks(range(num_tasks))
        ax.set_xticklabels(task_names)
        ax.set_ylim([0, 105])
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.show()


def plot_forgetting_matrix(accuracy_matrix, method_name, task_names=None):
    """Plot forgetting analysis heatmap."""
    num_tasks = accuracy_matrix.shape[0]
    if task_names is None:
        task_names = [f'Task {i+1}' for i in range(num_tasks)]
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(accuracy_matrix, annot=True, fmt='.1f', cmap='RdYlGn', 
                xticklabels=task_names, yticklabels=task_names,
                vmin=0, vmax=100, cbar_kws={'label': 'Accuracy (%)'})
    plt.xlabel('Test Task', fontsize=12)
    plt.ylabel('After Training on Task', fontsize=12)
    plt.title(f'Forgetting Analysis: {method_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def calculate_metrics(accuracy_matrix):
    """Calculate continual learning metrics."""
    num_tasks = accuracy_matrix.shape[0]
    
    # Average Accuracy
    avg_acc = accuracy_matrix[-1, :].mean()
    
    # Forgetting
    forgetting_values = []
    for task_id in range(num_tasks - 1):
        max_acc = accuracy_matrix[task_id, task_id]
        final_acc = accuracy_matrix[-1, task_id]
        forgetting = max_acc - final_acc
        forgetting_values.append(forgetting)
    
    avg_forgetting = np.mean(forgetting_values) if forgetting_values else 0.0
    
    return {'avg_accuracy': avg_acc, 'forgetting': avg_forgetting}

## 9. Experiment Configuration

In [ ]:
# Experiment hyperparameters
NUM_TASKS = 2
NUM_CLASSES = 2
CLASSES_PER_TASK = NUM_CLASSES // NUM_TASKS

ITERS_PER_TASK = 500
LEARNING_RATE = 0.001
BATCH_SIZE = 16

EWC_LAMBDA = 5000.0
BUFFER_SIZE_PER_CLASS = 50
FISHER_SAMPLES = 200

print("Experiment Configuration:")
print(f"  Tasks: {NUM_TASKS}")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Iterations/task: {ITERS_PER_TASK}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  EWC lambda: {EWC_LAMBDA}")
print(f"  Replay buffer/class: {BUFFER_SIZE_PER_CLASS}")

In [ ]:
# Create task datasets
print("\nCreating task datasets...")
train_task_datasets = create_task_datasets(train_dataset, NUM_TASKS)
test_task_datasets = create_task_datasets(test_dataset, NUM_TASKS)

## 10. Run Experiments

### 10.1 Baseline

In [ ]:
print("\n" + "="*60)
print("BASELINE EXPERIMENT")
print("="*60)

model_baseline = MedicalImageClassifier(num_classes=NUM_CLASSES, input_channels=3)
baseline_matrix = np.zeros((NUM_TASKS, NUM_TASKS))

for task_id in range(NUM_TASKS):
    print(f"\nTraining on Task {task_id + 1}...")
    train_baseline(model_baseline, train_task_datasets[task_id], 
                  ITERS_PER_TASK, LEARNING_RATE, BATCH_SIZE, device)
    
    print(f"\nEvaluating after Task {task_id + 1}:")
    for eval_task_id in range(NUM_TASKS):
        acc = evaluate_accuracy(model_baseline, test_task_datasets[eval_task_id], device)
        baseline_matrix[task_id, eval_task_id] = acc
        print(f"  Task {eval_task_id + 1}: {acc:.2f}%")

baseline_metrics = calculate_metrics(baseline_matrix)
print(f"\nBaseline Results:")
print(f"  Average Accuracy: {baseline_metrics['avg_accuracy']:.2f}%")
print(f"  Forgetting: {baseline_metrics['forgetting']:.2f}%")

### 10.2 EWC

In [ ]:
print("\n" + "="*60)
print("EWC EXPERIMENT")
print("="*60)

model_ewc = MedicalImageClassifier(num_classes=NUM_CLASSES, input_channels=3)
ewc_matrix = np.zeros((NUM_TASKS, NUM_TASKS))

for task_id in range(NUM_TASKS):
    print(f"\nTraining on Task {task_id + 1}...")
    train_with_ewc(model_ewc, train_task_datasets[task_id],
                  ITERS_PER_TASK, LEARNING_RATE, BATCH_SIZE, device,
                  current_task=task_id + 1, ewc_lambda=EWC_LAMBDA)
    
    # Estimate Fisher and store params
    if task_id < NUM_TASKS - 1:
        print(f"\nEstimating Fisher Information...")
        fisher_info = estimate_fisher_information(
            model_ewc, train_task_datasets[task_id], FISHER_SAMPLES, device
        )
        model_ewc.fisher_info = fisher_info
        
        optimal_params = {}
        for name, param in model_ewc.named_parameters():
            if param.requires_grad:
                optimal_params[name.replace('.', '__')] = param.data.clone()
        model_ewc.optimal_params = optimal_params
    
    print(f"\nEvaluating after Task {task_id + 1}:")
    for eval_task_id in range(NUM_TASKS):
        acc = evaluate_accuracy(model_ewc, test_task_datasets[eval_task_id], device)
        ewc_matrix[task_id, eval_task_id] = acc
        print(f"  Task {eval_task_id + 1}: {acc:.2f}%")

ewc_metrics = calculate_metrics(ewc_matrix)
print(f"\nEWC Results:")
print(f"  Average Accuracy: {ewc_metrics['avg_accuracy']:.2f}%")
print(f"  Forgetting: {ewc_metrics['forgetting']:.2f}%")

### 10.3 Experience Replay

In [ ]:
print("\n" + "="*60)
print("REPLAY EXPERIMENT")
print("="*60)

model_replay = MedicalImageClassifier(num_classes=NUM_CLASSES, input_channels=3)
replay_matrix = np.zeros((NUM_TASKS, NUM_TASKS))
memory_buffer = []

for task_id in range(NUM_TASKS):
    print(f"\nTraining on Task {task_id + 1}...")
    
    buffer_dataset = MemorySetDataset(memory_buffer) if len(memory_buffer) > 0 else None
    
    train_with_replay(model_replay, train_task_datasets[task_id],
                     ITERS_PER_TASK, LEARNING_RATE, BATCH_SIZE, device,
                     current_task=task_id + 1, buffer_dataset=buffer_dataset)
    
    # Update memory buffer
    print(f"\nUpdating memory buffer...")
    task_classes = list(range(task_id * CLASSES_PER_TASK, 
                              (task_id + 1) * CLASSES_PER_TASK))
    new_samples = fill_memory_buffer(
        train_task_datasets[task_id],
        BUFFER_SIZE_PER_CLASS,
        task_classes
    )
    memory_buffer.extend(new_samples)
    print(f"Buffer size: {len(memory_buffer)} samples")
    
    print(f"\nEvaluating after Task {task_id + 1}:")
    for eval_task_id in range(NUM_TASKS):
        acc = evaluate_accuracy(model_replay, test_task_datasets[eval_task_id], device)
        replay_matrix[task_id, eval_task_id] = acc
        print(f"  Task {eval_task_id + 1}: {acc:.2f}%")

replay_metrics = calculate_metrics(replay_matrix)
print(f"\nReplay Results:")
print(f"  Average Accuracy: {replay_metrics['avg_accuracy']:.2f}%")
print(f"  Forgetting: {replay_metrics['forgetting']:.2f}%")

### 10.4 EWC + Replay

In [ ]:
print("\n" + "="*60)
print("EWC + REPLAY EXPERIMENT")
print("="*60)

model_combined = MedicalImageClassifier(num_classes=NUM_CLASSES, input_channels=3)
combined_matrix = np.zeros((NUM_TASKS, NUM_TASKS))
memory_buffer_combined = []

for task_id in range(NUM_TASKS):
    print(f"\nTraining on Task {task_id + 1}...")
    
    buffer_dataset = MemorySetDataset(memory_buffer_combined) if len(memory_buffer_combined) > 0 else None
    
    train_with_ewc_replay(model_combined, train_task_datasets[task_id],
                         ITERS_PER_TASK, LEARNING_RATE, BATCH_SIZE, device,
                         current_task=task_id + 1, buffer_dataset=buffer_dataset,
                         ewc_lambda=EWC_LAMBDA)
    
    # Estimate Fisher and store params
    if task_id < NUM_TASKS - 1:
        print(f"\nEstimating Fisher Information...")
        fisher_info = estimate_fisher_information(
            model_combined, train_task_datasets[task_id], FISHER_SAMPLES, device
        )
        model_combined.fisher_info = fisher_info
        
        optimal_params = {}
        for name, param in model_combined.named_parameters():
            if param.requires_grad:
                optimal_params[name.replace('.', '__')] = param.data.clone()
        model_combined.optimal_params = optimal_params
    
    # Update memory buffer
    print(f"\nUpdating memory buffer...")
    task_classes = list(range(task_id * CLASSES_PER_TASK,
                              (task_id + 1) * CLASSES_PER_TASK))
    new_samples = fill_memory_buffer(
        train_task_datasets[task_id],
        BUFFER_SIZE_PER_CLASS,
        task_classes
    )
    memory_buffer_combined.extend(new_samples)
    print(f"Buffer size: {len(memory_buffer_combined)} samples")
    
    print(f"\nEvaluating after Task {task_id + 1}:")
    for eval_task_id in range(NUM_TASKS):
        acc = evaluate_accuracy(model_combined, test_task_datasets[eval_task_id], device)
        combined_matrix[task_id, eval_task_id] = acc
        print(f"  Task {eval_task_id + 1}: {acc:.2f}%")

combined_metrics = calculate_metrics(combined_matrix)
print(f"\nCombined Results:")
print(f"  Average Accuracy: {combined_metrics['avg_accuracy']:.2f}%")
print(f"  Forgetting: {combined_metrics['forgetting']:.2f}%")

## 11. Results Comparison

In [ ]:
# Compare all methods
results_comparison = {
    'Baseline': baseline_matrix[-1, :],
    'EWC': ewc_matrix[-1, :],
    'Replay': replay_matrix[-1, :],
    'EWC+Replay': combined_matrix[-1, :]
}

compare_methods(results_comparison)

In [ ]:
# Plot forgetting matrices
plot_forgetting_matrix(baseline_matrix, 'Baseline')
plot_forgetting_matrix(ewc_matrix, 'EWC')
plot_forgetting_matrix(replay_matrix, 'Replay')
plot_forgetting_matrix(combined_matrix, 'EWC+Replay')

In [ ]:
# Print summary table
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"{'Method':<20} {'Avg Accuracy':<15} {'Forgetting':<15}")
print("-"*80)

for method_name in ['Baseline', 'EWC', 'Replay', 'EWC+Replay']:
    if method_name == 'Baseline':
        metrics = baseline_metrics
    elif method_name == 'EWC':
        metrics = ewc_metrics
    elif method_name == 'Replay':
        metrics = replay_metrics
    else:
        metrics = combined_metrics
    
    print(f"{method_name:<20} {metrics['avg_accuracy']:>6.2f}%         {metrics['forgetting']:>6.2f}%")

print("="*80)

## 12. Conclusion

This notebook demonstrated:
- **Catastrophic Forgetting**: Baseline shows significant accuracy drop on previous tasks
- **EWC**: Reduces forgetting through parameter regularization
- **Replay**: Reduces forgetting by rehearsing stored samples
- **Combined**: Achieves best performance with minimal forgetting

### Key Findings:
- Continual learning is essential for medical imaging systems
- Combined approaches (EWC+Replay) perform best
- Trade-offs between memory, computation, and performance

### References:
- Kirkpatrick et al. (2017). "Overcoming catastrophic forgetting in neural networks." PNAS.
- van de Ven & Tolias (2019). "Three scenarios for continual learning." arXiv:1904.07734.